In [ ]:
import csv
import html
import json

from bugpipe import Bugpipe

# Exporting Data

This notebook demonstrates exporting issue data to CSV, JSON, and HTML using the standard library.

## Fetch Some Data

In [ ]:
with Bugpipe() as client:
    result = client.search("status:fixed type:vulnerability", page_size=50)

print(f"Fetched {len(result.issues)} issues")

## Build Row Dicts

Convert issues to a list of dicts for export. All fields are available directly on the `Issue` object.

In [ ]:
rows = []
for issue in result.issues:
    rows.append(
        {
            "id": issue.id,
            "status": issue.status.name,
            "priority": issue.priority.name,
            "title": issue.title,
            "owner": issue.owner,
            "reporter": issue.reporter,
            "type": issue.issue_type.name if issue.issue_type else None,
            "os": issue.os,
            "cve": issue.cve,
            "milestone": issue.milestone,
            "url": issue.url,
        }
    )

print(f"{len(rows)} rows ready for export")

## Export to CSV

In [ ]:
with open("issues.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("Saved to issues.csv")

## Export to JSON

In [ ]:
with open("issues.json", "w") as file:
    json.dump(rows, file, indent=4, default=str)

print("Saved to issues.json")

## Export to HTML

In [ ]:
headers = list(rows[0].keys())
lines = [
    "<!DOCTYPE html>",
    "<html><head><meta charset='utf-8'><title>issues</title></head><body>",
    "<table border='1' cellpadding='4' cellspacing='0'>",
    "<thead><tr><th>#</th>",
]
for header in headers:
    lines.append(f"  <th>{html.escape(header)}</th>")
lines.append("</tr></thead><tbody>")

for idx, row in enumerate(rows, start=1):
    lines.append(f"<tr><td>{idx}</td>")
    for header in headers:
        lines.append(f"  <td>{html.escape(str(row.get(header, '')))}</td>")
    lines.append("</tr>")

lines.append("</tbody></table></body></html>")

with open("issues.html", "w") as file:
    file.write("\n".join(lines))

print("Saved to issues.html")

## Filtering and Aggregating

Since issues are plain Python objects, you can filter and aggregate with list comprehensions and `collections.Counter`.

In [ ]:
from collections import Counter

# Filter by priority
p0_issues = [i for i in result.issues if i.priority.name == "P0"]
print(f"P0 issues: {len(p0_issues)}")
for issue in p0_issues[:5]:
    print(f"  #{issue.id} [{issue.status.name}] {issue.title}")

## Count by Status

In [ ]:
status_counts = Counter(i.status.name for i in result.issues)
for status, count in status_counts.most_common():
    print(f"  {status}: {count}")

## Count by Priority

In [ ]:
priority_counts = Counter(i.priority.name for i in result.issues)
for priority, count in sorted(priority_counts.items()):
    print(f"  {priority}: {count}")